# ANÁLISE DE MÉTRICAS - SPOTIFY
Samuel Pimenta, Vinícius Vilas Boas, Isabelly da Hora, Caio Mezini, Pedro Casarini 2°G

## Escolha da base de dados
Escolhemos uma base de dados do Spotify, que contém informações sobre músicas, como nome, artista, gênero, popularidade, duração, entre outros. A base de dados foi obtida através do Kaggle e possui mais de 100 mil registros de músicas.

[Spotify Tracks Genre Dataset](https://www.kaggle.com/datasets/thedevastator/spotify-tracks-genre-dataset)

## Colunas da base de dados
- artists: Nome(s) do(s) artista(s) associado(s) à faixa. (String)
- album_name: O nome do álbum ao qual a faixa pertence. (String)
- track_name: O nome da faixa. (String)
- popularity: A pontuação de popularidade da faixa no Spotify, variando de 0 a 100. (Integer)
- duration_ms: A duração da faixa em milissegundos. (Integer)
- explicit: Um valor booleano que indica se a faixa contém conteúdo explícito. (Boolean)
- danceability: Uma pontuação de 0 a 1 que representa o quão adequada uma faixa é para dançar com base em vários elementos musicais. (Float)
- energy: Uma medida da intensidade e atividade de uma faixa, variando de 0 a 1. (Float)
- key: A chave da faixa representada por um valor inteiro. (Integer)
- loudness: Os decibels da faixa (dB). (Float)
- mode: O modo tonal da faixa, representado por um valor inteiro (0 para menor, 1 para maior). (Integer)
- speechiness: Uma pontuação que varia de 0 a 1 e representa a presença de palavras faladas em uma faixa. (Float)
- acousticness: Uma pontuação que varia de 0 a 1 e representa o grau em que uma faixa possui uma qualidade acústica. (Float)
- instrumentalness: Uma pontuação que varia de 0 a 1 e representa a probabilidade de uma faixa ser instrumental. (Float)
- liveness: Uma pontuação que varia de 0 a 1 e representa a presença de um público durante a gravação ou performance de uma faixa. (Float)
- valence: Uma pontuação que varia de 0 a 1 e representa a positividade musical transmitida por uma faixa. (Float)
- tempo: O tempo da faixa em batidas por minuto (BPM). (Float)
- time_signature: O número de batidas dentro de cada barra da faixa. (Integer)
- track_genre: O gênero da faixa. (String)

## Tratamento e preparação dos dados

In [ ]:
import pandas as pd

df = spark.table('workspace.trabalho_bi.spotify_genre')
df_pandas = df.toPandas()

In [ ]:
df_pandas.head()

## Mudando nomes das colunas

In [ ]:
df_pandas = df_pandas.rename(
    columns={
        'Unnamed: 0':'nao_nomeado',
        'track_id':'id_musica',
        'artists':'artistas',
        'album_name':'nome_album',
        'track_name':'nome_musica',
        'popularity':'popularidade',
        'duration_ms':'duracao_ms',
        'explicit':'explicito',
        'danceability':'danceabilidade',
        'energy':'energia',
        'key':'tonalidade',
        'loudness':'nivel_barulho',
        'mode':'modo',
        'speechiness':'fala',
        'acousticness':'acustico',
        'instrumentalness':'instrumental',
        'liveness':'vivacidade',
        'valence':'valencia',
        'tempo':'tempo',
        'time_signature':'assinatura_tempo',
        'track_genre':'genero'
    }
)

In [ ]:
df_pandas.head()

In [ ]:
df_pandas.dtypes

## Contagem de valores nulos

In [ ]:
columns = ['nao_nomeado', 'id_musica', 'artistas', 'nome_album', 'nome_musica', 'popularidade', 'duracao_ms', 'explicito', 'danceabilidade', 'energia', 'tonalidade', 'nivel_barulho', 'modo', 'fala', 'acustico', 'instrumental', 'vivacidade', 'valencia', 'tempo', 'assinatura_tempo', 'genero']

nulos = df_pandas[columns].isnull().sum().reset_index()
nulos.columns = ['coluna', 'qnt_nulos']
nulos['pct_nulos'] = (nulos['qnt_nulos'] / len(df_pandas) * 100).round(2)

print(f"Total de linhas: {len(df_pandas)}\n")
print(nulos.to_string(index=False))

In [ ]:
colunas_verificar = ['artistas', 'nome_album', 'nome_musica']

df_nulos = df_pandas[df_pandas[colunas_verificar].isnull().any(axis=1)]

print(df_nulos)

In [ ]:
linha = df_pandas.iloc[65900]

for coluna, valor in linha.items():
    print(f"{coluna}: {valor}")

In [ ]:
df_pandas = df_pandas.drop(index=65900)

In [ ]:
nulos = df_pandas[columns].isnull().sum().reset_index()
nulos.columns = ['coluna', 'qnt_nulos']
nulos['pct_nulos'] = (nulos['qnt_nulos'] / len(df_pandas) * 100).round(2)

print(f"Total de linhas: {len(df_pandas)}\n")
print(nulos.to_string(index=False))

## Tratamento de valores nulos

In [ ]:
import random
import string

# ---- nao_nomeado: sequência crescente sem duplicar ----
ids_existentes = set(df['nao_nomeado'].dropna().astype(int).tolist())

def gerar_nao_nomeado():
    proximo = max(ids_existentes) + 1 if ids_existentes else 1
    while proximo in ids_existentes:
        proximo += 1
    ids_existentes.add(proximo)
    return proximo

mask_nao_nomeado = df_pandas['nao_nomeado'].isnull()
df_pandas.loc[mask_nao_nomeado, 'nao_nomeado'] = [gerar_nao_nomeado() for _ in range(mask_nao_nomeado.sum())]
df_pandas['nao_nomeado'] = df_pandas['nao_nomeado'].astype('int64')

# ---- id_musica: id aleatório único ----
ids_musica_existentes = set(df_pandas['id_musica'].dropna().tolist())

def gerar_id_musica(tamanho=22):
    caracteres = string.ascii_letters + string.digits
    while True:
        novo_id = ''.join(random.choices(caracteres, k=tamanho))
        if novo_id not in ids_musica_existentes:
            ids_musica_existentes.add(novo_id)
            return novo_id

mask_id_musica = df_pandas['id_musica'].isnull()
df_pandas.loc[mask_id_musica, 'id_musica'] = [gerar_id_musica() for _ in range(mask_id_musica.sum())]

# ---- demais colunas ----
padronizacao = {
    'artistas'        : 'Não Informado',
    'nome_album'      : 'Não Informado',
    'nome_musica'     : 'Não Informado',
    'popularidade'    : 0,
    'duracao_ms'      : 197000,
    'explicito'       : False,
    'danceabilidade'  : 0.5,
    'energia'         : 0.5,
    'tonalidade'      : 5,
    'nivel_barulho'   : -22.485,
    'modo'            : 1,
    'fala'            : 0.3,
    'acustico'        : 0.5,
    'instrumental'    : 0.01,
    'vivacidade'      : 0.3,
    'valencia'        : 0.5,
    'tempo'           : 121.0,
    'assinatura_tempo': 4,
    'genero'          : 'Não Informado'
}

for coluna, valor_padrao in padronizacao.items():
    df_pandas[coluna] = df_pandas[coluna].fillna(valor_padrao)

# ---- garantir tipos corretos ----
tipos = {
    'popularidade'    : 'int64',
    'duracao_ms'      : 'int64',
    'explicito'       : 'bool',
    'danceabilidade'  : 'float64',
    'energia'         : 'float64',
    'tonalidade'      : 'int64',
    'nivel_barulho'   : 'float64',
    'modo'            : 'int64',
    'fala'            : 'float64',
    'acustico'        : 'float64',
    'instrumental'    : 'float64',
    'vivacidade'      : 'float64',
    'valencia'        : 'float64',
    'tempo'           : 'float64',
    'assinatura_tempo': 'int64'
}

for coluna, tipo in tipos.items():
    df_pandas[coluna] = df_pandas[coluna].astype(tipo)

print("✅ Padronização concluída!")
print(f"Total de linhas: {len(df_pandas)}")
print(f"\nNulos restantes:\n{df_pandas.isnull().sum()[df_pandas.isnull().sum() > 0]}")

## Verificar tipos de dados

### Colunas Object

In [ ]:
# Verifica colunas object que têm valores não-string
colunas_object = ['id_musica', 'artistas', 'nome_album', 'nome_musica', 'genero']

print(f"{'COLUNA':<20} {'LINHAS COM TIPO ERRADO'}")
print("-" * 45)

for coluna in colunas_object:
    mask = df_pandas[coluna].apply(lambda x: not isinstance(x, str) and pd.notna(x))
    qtd = mask.sum()
    status = f"{qtd} linha(s)" if qtd > 0 else "✅ OK"
    print(f"{coluna:<20} {status}")

    if qtd > 0:
        print(df_pandas[mask][['nao_nomeado', coluna]].to_string(index=False))
        print()

### Colunas Float

In [ ]:
import numpy as np

colunas_float = ['danceabilidade', 'energia', 'nivel_barulho', 'fala', 'acustico', 'instrumental', 'vivacidade', 'valencia', 'tempo']

print(f"{'COLUNA':<20} {'LINHAS COM TIPO ERRADO'}")
print("-" * 45)

for coluna in colunas_float:
    mask = df_pandas[coluna].apply(lambda x: not isinstance(x, (float, np.floating)) and pd.notna(x))
    qtd = mask.sum()
    status = f"{qtd} linha(s)" if qtd > 0 else "✅ OK"
    print(f"{coluna:<20} {status}")

    if qtd > 0:
        print(df_pandas[mask][['nao_nomeado', coluna]].to_string(index=False))
        print()

### Colunas Int

In [ ]:
import numpy as np

colunas_int = ['nao_nomeado', 'popularidade', 'duracao_ms',
               'tonalidade', 'modo', 'assinatura_tempo']

print(f"{'COLUNA':<20} {'LINHAS COM TIPO ERRADO'}")
print("-" * 45)

for coluna in colunas_int:
    mask = df_pandas[coluna].apply(lambda x: not isinstance(x, (int, np.integer)) and pd.notna(x))
    qtd = mask.sum()
    status = f"{qtd} linha(s)" if qtd > 0 else "✅ OK"
    print(f"{coluna:<20} {status}")

    if qtd > 0:
        print(df_pandas[mask][['nao_nomeado', coluna]].to_string(index=False))
        print()

### Coluna Bool

In [ ]:
colunas_bool = ['explicito']

print(f"{'COLUNA':<20} {'LINHAS COM TIPO ERRADO'}")
print("-" * 45)

for coluna in colunas_bool:
    mask = df_pandas[coluna].apply(lambda x: not isinstance(x, bool) and pd.notna(x))
    qtd = mask.sum()
    status = f"{qtd} linha(s)" if qtd > 0 else "✅ OK"
    print(f"{coluna:<20} {status}")

    if qtd > 0:
        print(df_pandas[mask][['nao_nomeado', coluna]].to_string(index=False))
        print()

## Verificação de registros duplicados

In [ ]:
duplicados = df_pandas[df_pandas.duplicated(subset=['id_musica'], keep=False)]

print(f"Total de registros: {len(df_pandas)}")
print(f"Total de linhas duplicadas: {len(duplicados)}")
print(f"Total de id_musica duplicados: {df_pandas['id_musica'].duplicated().sum()}\n")

if len(duplicados) > 0:
    resultado = duplicados[['nao_nomeado', 'id_musica', 'nome_musica', 'artistas']]\
        .sort_values('id_musica')\
        .to_json(orient='records', indent=4)

    print(resultado)

## KPIs

### 1. Popularidade Média por Gênero
Fórmula: SOMA(popularidade) / TOTAL(faixas) agrupado por gênero

Importância: Identifica quais gêneros musicais possuem o maior apelo de público em massa no catálogo atual.

Interpretação: Gêneros com médias elevadas devem receber maior orçamento de marketing e destaque na página inicial. Valores baixos indicam nichos que podem ser despriorizados ou que precisam de estímulo promocional.

### 2. Taxa de Hits por Gênero (Hit Rate)
Fórmula: (TOTAL(faixas com popularidade >= 70) / TOTAL(faixas)) * 100 agrupado por gênero

Importância: Mostra a eficiência real de um gênero em produzir "superestrelas" ou faixas de extremo sucesso, indo além da média simples (que pode ser mascarada por muitas músicas medianas).

Interpretação: Uma taxa alta significa que o gênero é uma fábrica de sucessos comerciais (investimento altamente seguro). Uma taxa zero ou muito baixa indica um gênero estável, mas sem nenhum grande pico de engajamento no mercado.

### 3. Concentração de Popularidade (Top-10 Share)
Fórmula: (SOMA(popularidade dos 10 artistas mais populares) / SOMA(popularidade de todas as faixas)) * 100

Importância: Mede o nível de dependência ou risco de mercado do catálogo em relação a poucos artistas líderes.

Interpretação: Se o resultado for elevado (ex: acima de 40%), o catálogo corre risco de audiência (depende demais de pouca gente). Se for baixo (ex: menos de 15%), indica que a audiência está bem distribuída entre vários artistas, tornando a plataforma mais estável e diversa.

### 4. Índice de Potencial de Viralidade (Viral Score)
Fórmula: 0.40 * Popularidade_Normalizada + 0.20 * Danceabilidade_Normalizada + 0.15 * Energia_Normalizada + 0.15 * Valencia_Normalizada + 0.10 * Fala_Normalizada (calculado por faixa)

Importância: Cria uma nota única de algoritmo para rastrear músicas com os ingredientes perfeitos para viralizar em redes sociais (estilo Reels/TikTok).

Interpretação: Faixas com score perto de 1.0 são fortes candidatas a estouro orgânico e devem ser inseridas em playlists de recomendação automática. Scores baixos indicam faixas com perfil puramente acústico, lento ou sem tração comercial.

### 5. Percentual de Conteúdo Explícito por Gênero
Fórmula: (TOTAL(faixas onde explicito = True) / TOTAL(faixas)) * 100 agrupado por gênero

Importância: Essencial para a governança do catálogo, permitindo a segmentação de restrições de idade e curadoria de ambientes livres de palavrões.

Interpretação: Gêneros com alto percentual exigem filtros rigorosos de controle parental no app. Gêneros com taxa muito baixa ou zero são ideais para a criação de playlists "Família", comerciais ou trilhas para lojas físicas.

### 6. Fração de Instrumentalidade (Instrumental Share)
Fórmula: (TOTAL(faixas com instrumental >= 0.5) / TOTAL(faixas)) * 100 agrupado por gênero

Importância: Mapeia o volume de músicas sem vocal (focadas apenas em instrumentos ou batidas) dentro de cada estilo.

Interpretação: Gêneros com alto share instrumental são os pilares para a criação de playlists de foco, estudo (Lo-Fi, Ambient), meditação ou para licenciamento de trilhas sonoras de vídeos e jogos.

### 7. Duração Média das Faixas
Fórmula: (SOMA(duracao_ms) / TOTAL(faixas)) / 60000 agrupado por gênero

Importância: Permite entender o comportamento de extensão temporal do catálogo e tendências de consumo de tempo.

Interpretação: Uma média em queda ao longo do tempo reflete a tendência atual do streaming de criar faixas mais curtas para acelerar os "replays" e monetizar mais rápido. Gêneros com médias muito longas exigem mais tempo de retenção do usuário.

### 8. Percentual de Músicas Fora do Padrão Comercial
Fórmula: (TOTAL(faixas com duração < 3 min OU > 4 min) / TOTAL(faixas)) * 100

Importância: Avalia se o catálogo está alinhado com o padrão das rádios e das maiores playlists do mercado de streaming (que costumam preferir faixas de 3 a 4 minutos).

Interpretação: Uma taxa muito alta indica um catálogo alternativo ou conceitual (músicas longas demais ou vinhetas curtas demais). Para fins estritamente comerciais, quanto menor essa taxa, mais fácil é encaixar as faixas na programação padrão de reprodução.

### 9. Danceabilidade Média por Gênero
Fórmula: SOMA(danceabilidade) / TOTAL(faixas) agrupado por gênero

Importância: Isola a capacidade do ritmo de fazer o usuário se movimentar fisicamente.

Interpretação: Gêneros com médias próximas a 1.0 são os alvos perfeitos para alimentar categorias de playlists de "Festa", "Balada" ou "Treino/Academia".

### 10. Energia Média por Gênero
Fórmula: SOMA(energia) / TOTAL(faixas) agrupado por gênero

Importância: Mede a percepção de intensidade, velocidade e "barulho" das faixas do gênero.

Interpretação: Gêneros com alta energia indicam músicas rápidas e intensas (como Rock ou Eletrônica), enquanto valores baixos apontam para músicas calmas, acústicas ou de relaxamento. (A combinação deste indicador com o KPI 9 cria mapas visuais perfeitos para segmentar o humor do usuário).

In [ ]:
df_spark = spark.createDataFrame(df_pandas)

In [ ]:
from pyspark.sql.functions import avg, count, col, desc, lit, max, min, sum, when

In [ ]:
# TABELA 1: MÉTRICAS POR GÊNERO (df_kpis_por_genero)
# Agrupa os KPIs analíticos estruturados por estilo musical (KPIs 1, 2, 5, 6, 7, 8, 9 e 10)

df_kpis_por_genero = df_spark.groupBy("genero").agg(
    # KPI 1: Popularidade Média por Gênero
    avg("popularidade").alias("popularidade_media"),

    # KPI 2: Taxa de Hits por Gênero (Hit Rate)
    (sum(when(col("popularidade") >= 70, 1).otherwise(0)) / count("*")).alias("taxa_hits"),

    # KPI 5: Percentual de Conteúdo Explícito por Gênero
    (sum(when(col("explicito") == True, 1).otherwise(0)) / count("*")).alias("percentual_explicito"),

    # KPI 6: Fração de Instrumentalidade (Instrumental Share)
    (sum(when(col("instrumental") >= 0.5, 1).otherwise(0)) / count("*")).alias("percentual_instrumental"),

    # KPI 7: Duração Média das Faixas
    # Transforma o tempo de milissegundos para minutos para facilitar a leitura humana
    (avg("duracao_ms") / 60000).alias("duracao_media_minutos"),

    # KPI 8: Percentual de Músicas Fora do Padrão Comercial
    # Calcula a taxa de músicas que fogem do padrão de rádio/streaming (menores que 3 min ou maiores que 4 min)
    (sum(when((col("duracao_ms") / 60000 < 3) | (col("duracao_ms") / 60000 > 4), 1).otherwise(0)) / count("*")).alias("percentual_fora_padrao_duracao"),

    # KPI 9: Danceabilidade Média por Gênero
    # Isola o potencial rítmico do gênero para movimentação física (festas, treinos)
    avg("danceabilidade").alias("danceabilidade_media"),

    # KPI 10: Energia Média por Gênero
    # Isola o nível de intensidade, velocidade e percepção de "barulho" do estilo musical
    avg("energia").alias("energia_media"),

    # Métrica de Apoio Volumétrica
    # Essencial para calibrar o tamanho das bolhas no gráfico de dispersão (Scatter Plot)
    count("*").alias("total_faixas")
)

# Exibe o painel consolidado por gênero
display(df_kpis_por_genero)

In [ ]:
# TABELA 2: CONCENTRAÇÃO DE POPULARIDADE (df_concentracao_top_artistas)
# Isolamento do KPI 3 para análise de dependência e risco de portfólio

# Coleta da soma total de popularidade de todo o ecossistema (base para a divisão do Share)
total_popularidade_global = df_spark.select(sum("popularidade")).collect()[0][0]

# Criação do agrupamento focado no ranqueamento dos artistas dominantes
df_artistas_ranqueados = df_spark.groupBy("artistas") \
    .agg(
        sum("popularidade").alias("soma_popularidade_artista"),
        avg("popularidade").alias("media_popularidade_artista"),
        count("*").alias("quantidade_faixas")
    ) \
    .orderBy(desc("soma_popularidade_artista")) \
    .limit(10)

# KPI 3: Concentração de Popularidade (Top-10 Share)
# Extrai o peso que a soma dos 10 artistas mais populares tem sobre a plataforma inteira
soma_popularidade_top_10 = df_artistas_ranqueados.select(sum("soma_popularidade_artista")).collect()[0][0]
percentual_share_top_10 = (soma_popularidade_top_10 / total_popularidade_global) if total_popularidade_global > 0 else 0

# Adiciona o KPI como coluna fixa na tabela para alimentar facilmente o gráfico de rosca/donut
df_concentracao_top_artistas = df_artistas_ranqueados.withColumn("percentual_participacao_top_10", lit(percentual_share_top_10))

# Exibe o relatório de concentração de mercado
display(df_concentracao_top_artistas)

In [ ]:
# TABELA 3: INDEXAÇÃO DE PRODUTO (df_ranking_viral)
# Isolamento do KPI 4 para recomendação algorítmica profunda

# Limpeza e remoção de ruídos (registros sem gênero)
df_viral_filtrado = df_spark.filter((col("popularidade") > 0) & (col("genero") != 'Não Informado'))

# Mapeamento de mínimos e máximos absolutos necessários para a normalização Min-Max (escala 0 a 1)
limites_atributos = df_viral_filtrado.select(
    min("popularidade").alias("p_min"), max("popularidade").alias("p_max"),
    min("danceabilidade").alias("d_min"), max("danceabilidade").alias("d_max"),
    min("energia").alias("e_min"), max("energia").alias("e_max"),
    min("valencia").alias("v_min"), max("valencia").alias("v_max"),
    min("fala").alias("f_min"), max("fala").alias("f_max")
).collect()[0]

# KPI 4: Índice de Potencial de Viralidade (Viral Score)
# Equaliza as escalas das colunas e aplica a matriz de pesos (40% Pop, 20% Dance, 15% Energ, 15% Vale, 10% Fala)
df_ranking_viral = df_viral_filtrado \
    .withColumn("norm_popularidade", (col("popularidade") - limites_atributos["p_min"]) / (limites_atributos["p_max"] - limites_atributos["p_min"])) \
    .withColumn("norm_danceabilidade", (col("danceabilidade") - limites_atributos["d_min"]) / (limites_atributos["d_max"] - limites_atributos["d_min"])) \
    .withColumn("norm_energia", (col("energia") - limites_atributos["e_min"]) / (limites_atributos["e_max"] - limites_atributos["e_min"])) \
    .withColumn("norm_valencia", (col("valencia") - limites_atributos["v_min"]) / (limites_atributos["v_max"] - limites_atributos["v_min"])) \
    .withColumn("norm_fala", (col("fala") - limites_atributos["f_min"]) / (limites_atributos["f_max"] - limites_atributos["f_min"])) \
    .withColumn(
        "pontuacao_viral",
        (0.40 * col("norm_popularidade")) +
        (0.20 * col("norm_danceabilidade")) +
        (0.15 * col("norm_energia")) +
        (0.15 * col("norm_valencia")) +
        (0.10 * col("norm_fala"))
    ) \
    .select("nome_musica", "artistas", "genero", "popularidade", "danceabilidade", "pontuacao_viral") \
    .orderBy(desc("pontuacao_viral"))

# Exibe a listagem ordenada das faixas com maior propensão a viralizar
display(df_ranking_viral)